# Multi-task: encoder compartilhado entre toxicidade e inclinação política

**Atividade Prática — Deep Learning para Demandas Reais — fecha as hipóteses H2 e H3**

Aqui testamos a **arquitetura multi-task**: um **único encoder BERT compartilhado** com **duas
cabeças** (uma para toxicidade, outra para inclinação). Para isolar o efeito do compartilhamento,
treinamos o **mesmo encoder** (BERTimbau) em três configurações e comparamos a macro-F1 por tarefa:

1. **Single-task toxicidade** (encoder + 1 cabeça, só tweets);
2. **Single-task política** (encoder + 1 cabeça, só discursos);
3. **Multi-task** (encoder compartilhado + 2 cabeças, treino em round-robin).

Hipótese (H2/H3): como os registros são muito diferentes (tweet curto vs. discurso longo),
esperamos **transferência negativa** — o multi-task fica **abaixo** dos single-task.

> `Ambiente de execução → GPU`, depois `Executar tudo`.


## 1. Dependências e GPU

In [ ]:
!pip install -q -U transformers accelerate scikit-learn
import torch, numpy as np, pandas as pd
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)
SEED=42; torch.manual_seed(SEED); np.random.seed(SEED)

## 2. Dados de toxicidade (ToLD-Br) — subconjunto para o experimento

In [ ]:
!git clone -q https://github.com/JAugusto97/ToLD-Br.git
from sklearn.model_selection import train_test_split
CATS=["homophobia","obscene","insult","racism","misogyny","xenophobia"]
dt=pd.read_csv("ToLD-Br/ToLD-BR.csv").drop_duplicates(subset=["text"]).reset_index(drop=True)
dt["label"]=(dt[CATS].max(axis=1)>=2).astype(int)
# subconjunto para caber no tempo (mantendo proporcao)
dt,_=train_test_split(dt, train_size=10000, stratify=dt["label"], random_state=SEED)
tox_tr,tox_te=train_test_split(dt, test_size=0.2, stratify=dt["label"], random_state=SEED)
print("toxicidade treino/teste:", len(tox_tr), len(tox_te), "| %tox:", round(dt['label'].mean(),3))

## 3. Dados de política (Câmara) — coleta, limpeza, balanceamento, split por deputado

In [ ]:
import requests, time, random, re
API="https://dadosabertos.camara.leg.br/api/v2"; H={"Accept":"application/json"}
MAPA={"PT":0,"PSOL":0,"PCDOB":0,"PSB":0,"PDT":0,"REDE":0,"PL":1,"PP":1,"REPUBLICANOS":1,"UNIÃO":1,"NOVO":1,"PSC":1,"PSDB":1,"PSL":1,"PRTB":1}
def get(u):
    for i in range(4):
        try: return requests.get(u,headers=H,timeout=30).json()
        except Exception: time.sleep(1.5*(i+1))
    return {"dados":[],"links":[]}
deps=[]; url=f"{API}/deputados?idLegislatura=57&ordem=ASC&ordenarPor=nome&itens=100"
while url:
    d=get(url); deps+=d.get("dados",[]); url=next((l["href"] for l in d.get("links",[]) if l["rel"]=="next"),None)
random.seed(42); random.shuffle(deps); linhas=[]
for dep in deps:
    part=(dep.get("siglaPartido") or "").strip().upper()
    if part not in MAPA: continue
    n=0; u=f"{API}/deputados/{dep['id']}/discursos?itens=50&ordenarPor=dataHoraInicio&ordem=DESC&dataInicio=2023-02-01&dataFim=2024-12-31"
    while u and n<12:
        d=get(u)
        for disc in d.get("dados",[]):
            txt=(disc.get("transcricao") or "").strip()
            if len(txt.split())>=40:
                linhas.append({"texto":txt,"label":MAPA[part],"id_deputado":dep["id"]}); n+=1
                if n>=12: break
        u=next((l["href"] for l in d.get("links",[]) if l["rel"]=="next"),None)
    if len(linhas)>=3000: break
dp=pd.DataFrame(linhas)
SIG=r'\b(PT|PL|PP|PSOL|PCdoB|PCDOB|PSB|PDT|REDE|REPUBLICANOS|UNIÃO|UNIAO|NOVO|PSC|PSDB|PSL|PRTB|MDB|PSD)\b'
dp["texto"]=dp["texto"].map(lambda t: re.sub(SIG,'',re.sub(r'^.*?\)\s*-\s*','',str(t),count=1),flags=re.I))
nmin=dp["label"].value_counts().min()
dp=pd.concat([g.sample(nmin,random_state=42) for _,g in dp.groupby("label")]).reset_index(drop=True)
from sklearn.model_selection import GroupShuffleSplit
g=GroupShuffleSplit(1,test_size=0.2,random_state=42); tr_i,te_i=next(g.split(dp,dp["label"],groups=dp["id_deputado"]))
pol_tr,pol_te=dp.iloc[tr_i],dp.iloc[te_i]
print("politica treino/teste:", len(pol_tr), len(pol_te))

## 4. Tokenização, datasets e encoder compartilhado (BERTimbau)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
MODELO="neuralmind/bert-base-portuguese-cased"
tok=AutoTokenizer.from_pretrained(MODELO)
class TextDS(Dataset):
    def __init__(self, textos, labels, max_len):
        self.t=list(textos); self.y=list(labels); self.ml=max_len
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        e=tok(str(self.t[i]),truncation=True,max_length=self.ml,padding="max_length",return_tensors="pt")
        return {"input_ids":e["input_ids"].squeeze(0),"attention_mask":e["attention_mask"].squeeze(0),
                "labels":torch.tensor(int(self.y[i]))}
def loaders(frame_tr,frame_te,col_t,col_y,ml,bs=16):
    return (DataLoader(TextDS(frame_tr[col_t],frame_tr[col_y],ml),batch_size=bs,shuffle=True),
            DataLoader(TextDS(frame_te[col_t],frame_te[col_y],ml),batch_size=32))
tox_ld_tr,tox_ld_te=loaders(tox_tr,tox_te,"text","label",128)
pol_ld_tr,pol_ld_te=loaders(pol_tr,pol_te,"texto","label",256)
def pesos(y):
    y=np.array(y); n=len(y); n1=int(y.sum()); n0=n-n1
    return torch.tensor([n/(2*max(n0,1)),n/(2*max(n1,1))],dtype=torch.float).to(dev)
w_tox=pesos(tox_tr["label"]); w_pol=pesos(pol_tr["label"])
H=AutoModel.from_pretrained(MODELO).config.hidden_size
from sklearn.metrics import f1_score

## 5. Funções de treino (single-task e multi-task) e avaliação

In [ ]:
import itertools
EPOCHS=2; LR=2e-5

@torch.no_grad()
def avaliar(enc,head,drop,loader):
    enc.eval(); P=[]; Y=[]
    for b in loader:
        cls=enc(input_ids=b["input_ids"].to(dev),attention_mask=b["attention_mask"].to(dev)).last_hidden_state[:,0]
        P+=head(drop(cls)).argmax(-1).cpu().tolist(); Y+=b["labels"].tolist()
    return f1_score(Y,P,average="macro")

def treina_single(ld_tr,ld_te,w):
    enc=AutoModel.from_pretrained(MODELO).to(dev); head=nn.Linear(H,2).to(dev); drop=nn.Dropout(0.1)
    opt=torch.optim.AdamW(list(enc.parameters())+list(head.parameters()),lr=LR)
    ce=nn.CrossEntropyLoss(weight=w)
    for ep in range(EPOCHS):
        enc.train()
        for b in ld_tr:
            opt.zero_grad()
            cls=drop(enc(input_ids=b["input_ids"].to(dev),attention_mask=b["attention_mask"].to(dev)).last_hidden_state[:,0])
            ce(head(cls),b["labels"].to(dev)).backward(); opt.step()
    return avaliar(enc,head,drop,ld_te)

def treina_mtl(tox_tr,tox_te,pol_tr,pol_te,w_tox,w_pol):
    enc=AutoModel.from_pretrained(MODELO).to(dev)
    htox=nn.Linear(H,2).to(dev); hpol=nn.Linear(H,2).to(dev); drop=nn.Dropout(0.1)
    opt=torch.optim.AdamW(list(enc.parameters())+list(htox.parameters())+list(hpol.parameters()),lr=LR)
    ce_t=nn.CrossEntropyLoss(weight=w_tox); ce_p=nn.CrossEntropyLoss(weight=w_pol)
    for ep in range(EPOCHS):
        enc.train(); itp=itertools.cycle(pol_tr)
        for bt in tox_tr:
            bp=next(itp)
            # passo toxicidade (encoder + cabeca toxicidade)
            opt.zero_grad()
            cls=drop(enc(input_ids=bt["input_ids"].to(dev),attention_mask=bt["attention_mask"].to(dev)).last_hidden_state[:,0])
            ce_t(htox(cls),bt["labels"].to(dev)).backward(); opt.step()
            # passo politica (encoder + cabeca politica)
            opt.zero_grad()
            cls=drop(enc(input_ids=bp["input_ids"].to(dev),attention_mask=bp["attention_mask"].to(dev)).last_hidden_state[:,0])
            ce_p(hpol(cls),bp["labels"].to(dev)).backward(); opt.step()
    return avaliar(enc,htox,drop,tox_te), avaliar(enc,hpol,drop,pol_te)

## 6. Rodar os três experimentos

In [ ]:
import time
t0=time.time()
print("treinando single-task toxicidade..."); st_tox=treina_single(tox_ld_tr,tox_ld_te,w_tox)
print("  macro-F1:",round(st_tox,4))
print("treinando single-task politica..."); st_pol=treina_single(pol_ld_tr,pol_ld_te,w_pol)
print("  macro-F1:",round(st_pol,4))
print("treinando multi-task (compartilhado)..."); mt_tox,mt_pol=treina_mtl(tox_ld_tr,tox_ld_te,pol_ld_tr,pol_ld_te,w_tox,w_pol)
print("  macro-F1 tox/pol:",round(mt_tox,4),round(mt_pol,4))
print("tempo total (min):",round((time.time()-t0)/60,1))

## 7. Comparação, gráfico e conclusão sobre H2/H3

In [ ]:
import json, matplotlib.pyplot as plt
res={"single_tox":round(st_tox,4),"single_pol":round(st_pol,4),
     "multi_tox":round(mt_tox,4),"multi_pol":round(mt_pol,4),
     "delta_tox":round(mt_tox-st_tox,4),"delta_pol":round(mt_pol-st_pol,4),
     "encoder":MODELO}
print(json.dumps(res,indent=2,ensure_ascii=False))
json.dump(res,open("resultados_multitask.json","w"),ensure_ascii=False,indent=2)

import numpy as np
labels=["Toxicidade","Política"]; x=np.arange(2); w=0.35
plt.figure(figsize=(6,4))
plt.bar(x-w/2,[st_tox,st_pol],w,label="Single-task",color="#5b4bdb")
plt.bar(x+w/2,[mt_tox,mt_pol],w,label="Multi-task",color="#c0392b")
plt.xticks(x,labels); plt.ylabel("macro-F1"); plt.ylim(0,1); plt.legend()
plt.title("Single-task vs Multi-task (mesmo encoder BERTimbau)")
for i,(a,b) in enumerate(zip([st_tox,st_pol],[mt_tox,mt_pol])):
    plt.text(i-w/2,a+0.01,f"{a:.2f}",ha="center"); plt.text(i+w/2,b+0.01,f"{b:.2f}",ha="center")
plt.tight_layout(); plt.savefig("fig_multitask.png",dpi=150); plt.show()

veredito = ("NEGATIVA (multi-task abaixo): H2/H3 confirmadas" if (mt_tox<st_tox and mt_pol<st_pol)
            else "MISTA/POSITIVA: ver deltas")
print("Transferencia:", veredito)

**Leitura.** Se o multi-task ficar **abaixo** dos single-task nas duas tarefas (deltas
negativos), confirma-se a **transferência negativa** (H2): um encoder compartilhado entre os dois
registros não consegue ser ótimo para ambos. Junto com os achados anteriores (BERT vence só na
toxicidade; janelamento recupera a política), isso fecha a tese de que **registros distintos
exigem modelos distintos** — encerrando H2 e H3 com evidência experimental.

In [ ]:
# Baixar os resultados (rode esta celula apos os experimentos; nao retreina)
try:
    from google.colab import files
    files.download('resultados_multitask.json')
    files.download('fig_multitask.png')
except Exception as e:
    print('Use o painel de Arquivos a esquerda para baixar os 2 arquivos.', e)